# 🎬 AI Content Automation Pipeline
**Idea → Script → [Voiceover + B-roll] → Assembly → YouTube Upload**

Stack: Pure Python · Anthropic Claude · Kokoro TTS · Pexels API · FFmpeg · YouTube Data API v3

---
Run cells top-to-bottom for a full pipeline run, or execute individual sections for debugging.

## 0 · Setup — Install Dependencies

In [1]:
# Install required packages (run once)
import sys
!{sys.executable} -m pip install -q \
    anthropic \
    python-dotenv \
    requests \
    kokoro \
    soundfile \
    google-api-python-client \
    google-auth-httplib2 \
    google-auth-oauthlib \
    google-genai

print('✅ Dependencies installed')

✅ Dependencies installed


## 1 · Config & Logging

In [2]:
# ─── config.py (inlined) ─────────────────────────────────────────────────────
import os
import logging
import json
import csv
import time
import traceback
import subprocess
import requests
import concurrent.futures
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('pipeline.env')   # adjust path if .env lives elsewhere

# ── API Keys ─────────────────────────────────────────────────
GEMINI_API_KEY            = os.getenv('Gemini_API_KEY', '')
ANTHROPIC_API_KEY         = os.getenv('ANTHROPIC_API_KEY', '')  # unused; kept for diagnostics compatibility
PEXELS_API_KEY               = os.getenv('PEXELS_API_KEY', '')
PIXABAY_API_KEY              = os.getenv('PIXABAY_API_KEY', '')

# ── YouTube ──────────────────────────────────────────────────
YOUTUBE_CLIENT_SECRETS_PATH  = os.getenv('YOUTUBE_CLIENT_SECRETS_PATH',
                                          'credentials/client_secrets.json')
YOUTUBE_TOKEN_PATH           = os.getenv('YOUTUBE_TOKEN_PATH',
                                          'credentials/youtube_token.json')
YOUTUBE_PRIVACY_STATUS       = os.getenv('YOUTUBE_PRIVACY_STATUS', 'unlisted')

# ── Kokoro TTS ───────────────────────────────────────────────
KOKORO_VOICE                 = os.getenv('KOKORO_VOICE', 'af_heart')
KOKORO_OUTPUT_FORMAT         = os.getenv('KOKORO_OUTPUT_FORMAT', 'mp3')

# ── Pipeline inputs ──────────────────────────────────────────
TOPIC_SEED                   = os.getenv('TOPIC_SEED', 'productivity habits for developers')
CONTENT_NICHE                = os.getenv('CONTENT_NICHE', 'tech lifestyle')
TARGET_AUDIENCE              = os.getenv('TARGET_AUDIENCE', 'software developers aged 22-35')

# ── Directories ──────────────────────────────────────────────
OUTPUT_DIR                   = Path(os.getenv('OUTPUT_DIR',   'output'))
CLIPS_DIR                    = Path(os.getenv('CLIPS_DIR',    'output/clips'))
FALLBACK_DIR                 = Path(os.getenv('FALLBACK_DIR', 'assets/fallback'))
CREDENTIALS_DIR              = Path(os.getenv('CREDENTIALS_DIR', 'credentials'))
LOGS_DIR                     = Path(os.getenv('LOGS_DIR',     'logs'))

# Create directories
for d in [OUTPUT_DIR, CLIPS_DIR, FALLBACK_DIR, CREDENTIALS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Logging ──────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(LOGS_DIR / 'pipeline_log.txt'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger('pipeline')

TS = datetime.now().strftime('%Y%m%d_%H%M%S')  # shared timestamp for this run

print(f'✅ Config loaded  |  Run timestamp: {TS}')
print(f'   Topic   : {TOPIC_SEED}')
print(f'   Niche   : {CONTENT_NICHE}')
print(f'   Audience: {TARGET_AUDIENCE}')

✅ Config loaded  |  Run timestamp: 20260610_083137
   Topic   : productivity habits for developers
   Niche   : tech lifestyle
   Audience: software developers aged 22-35


In [3]:
import sys

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

## 2 · Helpers — Retry, JSON Parsing, Logging

In [4]:
from google import genai
import time

client = genai.Client(api_key=GEMINI_API_KEY)

# ─── Retry wrapper ────────────────────────────────────────────
def call_gemini(prompt, system='', temperature=0.7, max_retries=3):
    """Call Gemini with exponential-backoff retry."""
    for attempt in range(1, max_retries + 1):
        try:
            logger.info(
                f'Gemini API call (attempt {attempt}) temperature={temperature}'
            )

            full_prompt = f"{system}\n\n{prompt}" if system else prompt

            response = client.models.generate_content(
                model="gemini-2.0-flash",
                contents=full_prompt,
                config={
                    "temperature": temperature,
                    "max_output_tokens": 2048,
                },
            )

            return response.text

        except Exception as e:
            wait = 2 ** attempt
            logger.warning(
                f'Gemini API error (attempt {attempt}): {e}. Retrying in {wait}s...'
            )
            log_error('gemini_api', e)

            if attempt == max_retries:
                raise

            time.sleep(wait)
# ─── JSON extraction ─────────────────────────────────────────
def extract_json(raw: str) -> dict:
    """Strip markdown fences and parse JSON robustly."""
    cleaned = raw.strip()
    for fence in ['```json', '```JSON', '```']:
        if cleaned.startswith(fence):
            cleaned = cleaned[len(fence):]
    if cleaned.endswith('```'):
        cleaned = cleaned[:-3]
    return json.loads(cleaned.strip())

# ─── Error logger ────────────────────────────────────────────
def log_error(step: str, exc: Exception):
    with open(LOGS_DIR / 'error_log.txt', 'a') as f:
        f.write(f'\n[{datetime.now().isoformat()}] STEP={step}\n')
        f.write(traceback.format_exc())
        f.write('\n' + '─' * 60 + '\n')

# ─── Step timer decorator ────────────────────────────────────
def timed_step(name):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            logger.info(f'START  {name}')
            t0 = time.time()
            result = fn(*args, **kwargs)
            elapsed = round(time.time() - t0, 2)
            logger.info(f'FINISH {name}  ({elapsed}s)')
            return result
        return wrapper
    return decorator

print('✅ Helpers ready')

✅ Helpers ready


## 3 · Step 1A — Idea Generation

In [5]:
import json
import time
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)


def call_gemini(prompt: str,
                system: str = "",
                temperature: float = 0.8,
                max_retries: int = 3) -> str:
    """Call Gemini and return raw text."""

    full_prompt = f"{system}\n\n{prompt}" if system else prompt

    for attempt in range(1, max_retries + 1):
        try:
            logger.info(
                f"Gemini API call (attempt {attempt}) "
                f"temperature={temperature}"
            )

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=full_prompt,
                config={
                    "temperature": temperature,
                    "max_output_tokens": 2048,
                    "response_mime_type": "application/json",
                },
            )

            return response.text

        except Exception as e:
            error_msg = str(e)

            logger.warning(
                f"Gemini API error (attempt {attempt}): {error_msg}"
            )

            # Don't retry invalid keys
            if "API_KEY_INVALID" in error_msg:
                raise RuntimeError(
                    "Invalid or expired Gemini API key."
                )

            # Don't retry quota errors endlessly
            if "RESOURCE_EXHAUSTED" in error_msg:
                raise RuntimeError(
                    "Gemini quota exceeded."
                )

            if attempt == max_retries:
                raise

            wait = 2 ** attempt
            time.sleep(wait)


@timed_step("Step 1A - Idea Generation")
def generate_idea(
    topic: str,
    niche: str,
    audience: str
) -> dict:

    system = (
        "You are an expert short-form video strategist. "
        "Return ONLY valid JSON."
    )

    prompt = f"""
Generate a short-form video idea.

Topic: {topic}
Niche: {niche}
Audience: {audience}

Return ONLY this JSON schema:

{{
  "title": "",
  "hook": "",
  "key_points": ["", "", ""],
  "cta": "",
  "keywords": ["", "", "", ""]
}}

Rules:
- title under 60 characters
- hook must be curiosity-driven
- exactly 3 key points
- each key point is one spoken sentence
- 3 to 5 keywords
- valid JSON only
"""

    raw = call_gemini(
        prompt=prompt,
        system=system,
        temperature=0.8
    )

    idea = json.loads(raw)

    idea_path = OUTPUT_DIR / f"idea_{TS}.json"

    with open(idea_path, "w", encoding="utf-8") as f:
        json.dump(
            idea,
            f,
            indent=2,
            ensure_ascii=False
        )

    logger.info(f"Idea saved -> {idea_path}")

    return idea


# Run
idea = generate_idea(
    TOPIC_SEED,
    CONTENT_NICHE,
    TARGET_AUDIENCE
)

print(json.dumps(
    idea,
    indent=2,
    ensure_ascii=False
))

2026-06-10 08:31:40,860 [INFO] START  Step 1A - Idea Generation
2026-06-10 08:31:40,861 [INFO] Gemini API call (attempt 1) temperature=0.8
2026-06-10 08:31:40,863 [INFO] AFC is enabled with max remote calls: 10.
2026-06-10 08:31:45,179 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 08:31:45,185 [INFO] Idea saved -> output\idea_20260610_083137.json
2026-06-10 08:31:45,190 [INFO] FINISH Step 1A - Idea Generation  (4.33s)
{
  "title": "Dev Productivity: My Top 3 Habits for Focus!",
  "hook": "Feeling overwhelmed by your codebase? What if I told you there are 3 simple habits that can transform your daily coding?",
  "key_points": [
    "Implement the Pomodoro Technique to break down your coding sessions into focused sprints.",
    "Start your day by tackling the most challenging task first, known as 'Eat the Frog'.",
    "Leverage keyboard shortcuts and automation tools to minimize repetitive a

## 4 · Step 1B — Script Generation

In [6]:
@timed_step('Step 1B — Script Generation')
def generate_script(idea: dict) -> dict:
    system = (
        'You are a professional short-form video scriptwriter. '
        'Return ONLY valid JSON — no markdown, no explanation.'
    )
    prompt = f"""Write a narration script for this short-form video idea:

{json.dumps(idea, indent=2)}

Return a JSON object with EXACTLY these keys:
{{
  "script"           : "<full narration as plain spoken text>",
  "duration_estimate": <integer seconds, target 60-90>,
  "segments": [
    {{ "label": "hook",    "text": "<hook narration>" }},
    {{ "label": "point_1", "text": "<key point 1 narration>" }},
    {{ "label": "point_2", "text": "<key point 2 narration>" }},
    {{ "label": "point_3", "text": "<key point 3 narration>" }},
    {{ "label": "cta",     "text": "<CTA narration>" }}
  ]
}}

Rules:
- Total word count: 150-225 words (60-90 seconds of speech)
- Tone: natural, conversational — write exactly as spoken into a mic
- NEVER use: 'In this video', 'As we know', "It's important to note"
- No markdown, no stage directions, no parenthetical notes
- Each segment ends at a natural breath pause"""

    raw = call_gemini(prompt=prompt,
                      system=system, temperature=0.5)
    script_data = extract_json(raw)

    # Save plain-text script
    script_path = OUTPUT_DIR / f'script_{TS}.txt'
    script_path.write_text(script_data['script'])
    logger.info(f'Script saved → {script_path}')

    # Save full JSON too
    (OUTPUT_DIR / f'script_{TS}.json').write_text(json.dumps(script_data, indent=2))
    return script_data

# ── Run ──────────────────────────────────────────────────────
script_data = generate_script(idea)
print(f"Duration estimate: {script_data['duration_estimate']}s")
print(f"Word count: {len(script_data['script'].split())}")
print('\n--- SCRIPT ---')
print(script_data['script'])

2026-06-10 08:31:45,240 [INFO] START  Step 1B — Script Generation
2026-06-10 08:31:45,249 [INFO] Gemini API call (attempt 1) temperature=0.5
2026-06-10 08:31:45,260 [INFO] AFC is enabled with max remote calls: 10.
2026-06-10 08:31:52,439 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 08:31:52,455 [INFO] Script saved → output\script_20260610_083137.txt
2026-06-10 08:31:52,506 [INFO] FINISH Step 1B — Script Generation  (7.26s)
Duration estimate: 70s
Word count: 147

--- SCRIPT ---
Feeling overwhelmed by your codebase? What if I told you there are three simple habits that can totally transform your daily coding focus? First up, the Pomodoro Technique. Break your coding sessions into focused sprints. Work intensely for 25 minutes, then take a 5-minute break. This keeps your mind sharp and prevents burnout. Next, 'Eat the Frog'. Start your day by tackling your most challenging task first. Get th

--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 44: character maps to <undefined>
Call stack:
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\deven\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\deven\AppData\Roaming\Python\Python310\site

## 5 · Step 2A — Voiceover (Kokoro TTS)

In [7]:
@timed_step('Step 2A — Kokoro TTS')
def generate_voiceover(script_data: dict) -> Path:
    """
    Generate voiceover using Kokoro TTS (local, no API cost).
    Returns path to audio file. Halts pipeline on any failure.
    """
    full_text = ' '.join(seg['text'] for seg in script_data['segments'])
    audio_path = OUTPUT_DIR / f'audio_{TS}.{KOKORO_OUTPUT_FORMAT}'

    try:
        from kokoro import KPipeline
        import soundfile as sf
        import numpy as np

        logger.info(f'Loading Kokoro pipeline, voice={KOKORO_VOICE}')
        pipeline = KPipeline(lang_code='a')   # 'a' = American English

        audio_chunks = []
        for _, _, audio in pipeline(full_text, voice=KOKORO_VOICE, speed=1.0, split_pattern=r'\n+'):
            audio_chunks.append(audio)

        combined = np.concatenate(audio_chunks)
        sf.write(str(audio_path), combined, samplerate=24000)
        logger.info(f'Voiceover saved → {audio_path}')
        return audio_path

    except ImportError:
        # ── Dev fallback: generate a silent placeholder WAV ─────────
        logger.warning('Kokoro not installed — generating silent placeholder audio')
        try:
            import soundfile as sf
            import numpy as np
            silence = np.zeros(int(24000 * script_data['duration_estimate']), dtype=np.float32)
            audio_path = audio_path.with_suffix('.wav')
            sf.write(str(audio_path), silence, samplerate=24000)
            logger.info(f'Placeholder audio saved → {audio_path}')
            return audio_path
        except Exception as e:
            log_error('kokoro_tts', e)
            raise RuntimeError('TTS failed — halting pipeline') from e

    except Exception as e:
        log_error('kokoro_tts', e)
        raise RuntimeError('Kokoro TTS failed — halting pipeline') from e

# ── Run (will be called inside parallel block below) ─────────
# audio_path = generate_voiceover(script_data)
print('✅ Step 2A function defined')

✅ Step 2A function defined


## 6 · Step 2B — Video Clips (Pexels + Pixabay)

In [8]:
# ─── Step 2B — Video Clips (Pexels API + Coverr no-key fallback) ─────────────
#
#  Source priority:
#   1. Pexels API  (free key from pexels.com/api/new — takes 30 seconds)
#   2. Coverr      (completely free, NO API key, NO signup)
#   3. Local       assets/fallback/*.mp4
# ─────────────────────────────────────────────────────────────────────────────

_pexels_request_times = []
PEXELS_MAX_PER_HOUR   = 200

def _pexels_rate_guard():
    now = time.time()
    hour_ago = now - 3600
    _pexels_request_times[:] = [t for t in _pexels_request_times if t > hour_ago]
    if len(_pexels_request_times) >= PEXELS_MAX_PER_HOUR:
        wait = 3600 - (now - _pexels_request_times[0])
        logger.warning(f'Pexels rate limit reached -- sleeping {wait:.0f}s')
        time.sleep(wait)
    _pexels_request_times.append(time.time())


def _download_clip(url: str, dest: Path) -> bool:
    """Stream-download a video file."""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        r = requests.get(url, stream=True, timeout=60, headers=headers)
        r.raise_for_status()
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
        return True
    except Exception as e:
        logger.warning(f'Download failed {url}: {e}')
        return False


def _fetch_pexels(keyword: str) -> str | None:
    """Return a video URL from Pexels or None."""
    if not PEXELS_API_KEY or PEXELS_API_KEY == 'YOUR_PEXELS_KEY_HERE':
        return None
    _pexels_rate_guard()
    try:
        r = requests.get(
            'https://api.pexels.com/videos/search',
            headers={'Authorization': PEXELS_API_KEY},
            params={'query': keyword, 'per_page': 1, 'orientation': 'portrait'},
            timeout=15
        )
        r.raise_for_status()
        videos = r.json().get('videos', [])
        if not videos:
            return None
        files = sorted(videos[0].get('video_files', []),
                       key=lambda f: f.get('height', 0), reverse=True)
        for f in files:
            if f.get('height', 9999) <= 1080:
                return f['link']
        return files[0]['link'] if files else None
    except Exception as e:
        logger.warning(f'Pexels error for "{keyword}": {e}')
        return None


def _fetch_coverr(keyword: str) -> str | None:
    try:
        r = requests.get(
            'https://coverr.co/api/videos',
            params={
                'keywords': keyword,
                'page':     1,
                'language': 'en'
            },
            headers={
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
                'Accept':     'application/json',
                'Referer':    'https://coverr.co/'
            },
            timeout=15
        )
        r.raise_for_status()
        data = r.json()

        # Print full first hit so we can see the real schema
        hits = data.get('hits', [])
        if not hits:
            logger.warning(f'Coverr: no hits for "{keyword}"')
            return None

        first = hits[0]
        print(f'Coverr first hit keys: {list(first.keys())}')  # debug line

        # Try every known key pattern
        url = (
            first.get('mp4_url')
            or first.get('file_url')
            or first.get('coverr_url')
            or first.get('url')
            or (first.get('sources') or {}).get('mp4')
            or (first.get('urls') or {}).get('mp4')
            or (first.get('video') or {}).get('url')
        )

        if not url:
            print(f'Coverr hit sample: {str(first)[:400]}')  # debug
        return url

    except Exception as e:
        logger.warning(f'Coverr error for "{keyword}": {e}')
        return None
    
def _get_fallback_clip() -> Path | None:
    clips = list(FALLBACK_DIR.glob('*.mp4'))
    if clips:
        return clips[0]
    logger.warning('No fallback clips found in assets/fallback/')
    return None


@timed_step('Step 2B -- Video Clips')
def download_clips(keywords: list[str]) -> list[Path]:
    downloaded = []
    for kw in keywords:
        safe_kw = kw.replace(' ', '_')
        dest = CLIPS_DIR / f'clip_{safe_kw}_{TS}.mp4'
        url, source = None, None

        # 1. Pexels
        url = _fetch_pexels(kw)
        if url:
            source = 'pexels'

        # 2. Coverr (no key needed)
        if not url:
            logger.warning(f'Pexels miss for "{kw}" -- trying Coverr (no key)')
            url = _fetch_coverr(kw)
            if url:
                source = 'coverr'

        # 3. Local fallback
        if url and _download_clip(url, dest):
            logger.info(f'Clip downloaded ({source}) -> {dest}')
            downloaded.append(dest)
        else:
            fb = _get_fallback_clip()
            if fb:
                logger.warning(f'Using local fallback clip for "{kw}"')
                downloaded.append(fb)
            else:
                logger.error(f'No clip available for "{kw}" -- skipping')

    return downloaded

print('Step 2B functions defined')
print(f'  Pexels key : {"SET" if PEXELS_API_KEY and PEXELS_API_KEY != "YOUR_PEXELS_KEY_HERE" else "NOT SET -- will skip to Coverr"}')
print(f'  Coverr     : no key required -- always available as fallback')

Step 2B functions defined
  Pexels key : SET
  Coverr     : no key required -- always available as fallback


## 7 · Steps 2A + 2B — Parallel Execution

In [9]:

def _create_fallback_clip() -> Path:
    """Generate a solid black fallback clip using FFmpeg when no clips are available."""
    fallback_path = OUTPUT_DIR / f'fallback_clip_{TS}.mp4'
    try:
        subprocess.run([
            'ffmpeg', '-y',
            '-f', 'lavfi', '-i', 'color=c=black:s=1080x1920:r=30',
            '-t', '30',
            '-c:v', 'libx264', '-crf', '23', '-preset', 'fast',
            str(fallback_path)
        ], capture_output=True, text=True, check=True)
        logger.info(f'Fallback clip created -> {fallback_path}')
    except Exception as e:
        logger.error(f'Could not create fallback clip: {e}')
        raise RuntimeError('No clips available and fallback clip creation failed') from e
    return fallback_path

audio_path = None
clip_paths = []
errors     = {}

logger.info('START Parallel -- TTS + Clip Downloads')
t0 = time.time()

# Submit all jobs first simultaneously
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    future_audio = executor.submit(generate_voiceover, script_data)
    future_clips = executor.submit(download_clips, idea['keywords'])

    # Wait for BOTH with generous timeout (video gen can take 5+ mins)
    done, not_done = concurrent.futures.wait(
        [future_audio, future_clips],
        timeout=600,   # 10 minute ceiling for both
        return_when=concurrent.futures.ALL_COMPLETED
    )

    # Cancel anything still running after timeout
    for f in not_done:
        f.cancel()
        logger.warning('A future hit the 10min timeout and was cancelled')

    # Collect audio result
    if future_audio in done:
        try:
            audio_path = future_audio.result()
            print(f'Audio ready -> {audio_path}')
        except Exception as e:
            errors['tts'] = e
            print(f'TTS failed: {e}')
    else:
        errors['tts'] = TimeoutError('TTS timed out')
        print('TTS timed out')

    # Collect clips result
    if future_clips in done:
        try:
            clip_paths = future_clips.result()
            print(f'Clips ready -> {len(clip_paths)} downloaded')
            for p in clip_paths:
                print(f'  {p}')
        except Exception as e:
            errors['clips'] = e
            print(f'Clips failed: {e}')
    else:
        errors['clips'] = TimeoutError('Clip generation timed out')
        print('Clip generation timed out')

elapsed = round(time.time() - t0, 2)
logger.info(f'FINISH Parallel ({elapsed}s)')

# Critical gate
if 'tts' in errors or audio_path is None:
    raise RuntimeError('Audio missing -- pipeline halted. Check error_log.txt')

if not clip_paths:
    logger.warning('No clips downloaded -- using fallback solid-color clip')
    clip_paths = [_create_fallback_clip()]   # generate one programmatically

print(f'Parallel phase complete in {elapsed}s')

2026-06-10 08:31:52,684 [INFO] START Parallel -- TTS + Clip Downloads
2026-06-10 08:31:52,693 [INFO] START  Step 2A — Kokoro TTS
2026-06-10 08:31:52,702 [INFO] START  Step 2B -- Video Clips
2026-06-10 08:31:53,705 [INFO] Clip downloaded (pexels) -> output\clips\clip_dev_productivity_20260610_083137.mp4
2026-06-10 08:31:54,199 [INFO] Clip downloaded (pexels) -> output\clips\clip_coding_tips_20260610_083137.mp4
2026-06-10 08:31:55,242 [INFO] Clip downloaded (pexels) -> output\clips\clip_developer_habits_20260610_083137.mp4
2026-06-10 08:31:56,047 [INFO] Clip downloaded (pexels) -> output\clips\clip_tech_lifestyle_20260610_083137.mp4
2026-06-10 08:31:56,798 [INFO] Clip downloaded (pexels) -> output\clips\clip_software_engineer_20260610_083137.mp4
2026-06-10 08:31:56,798 [INFO] FINISH Step 2B -- Video Clips  (4.09s)
2026-06-10 08:32:14,440 [INFO] Loading Kokoro pipeline, voice=af_heart
2026-06-10 08:32:14,751 [INFO] HTTP Request: HEAD https://huggingface.co/hexgrad/Kokoro-82M/resolve/main/

c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\rnn.py:1009: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


2026-06-10 08:32:16,102 [INFO] HTTP Request: HEAD https://huggingface.co/hexgrad/Kokoro-82M/resolve/main/kokoro-v1_0.pth "HTTP/1.1 302 Found"


2026-06-10 08:32:16,105 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-10 08:32:19,065 [INFO] HTTP Request: HEAD https://huggingface.co/hexgrad/Kokoro-82M/resolve/main/voices/af_heart.pt "HTTP/1.1 302 Found"
2026-06-10 08:32:38,763 [INFO] Voiceover saved → output\audio_20260610_083137.mp3
2026-06-10 08:32:38,802 [INFO] FINISH Step 2A — Kokoro TTS  (46.1s)
Audio ready -> output\audio_20260610_083137.mp3
Clips ready -> 5 downloaded
  output\clips\clip_dev_productivity_20260610_083137.mp4
  output\clips\clip_coding_tips_20260610_083137.mp4
  output\clips\clip_developer_habits_20260610_083137.mp4
  output\clips\clip_tech_lifestyle_20260610_083137.mp4
  output\clips\clip_software_engineer_20260610_083137.mp4
2026-06-10 08:32:38,804 [INFO] FINISH Parallel (46.11s)
Parallel phase complete in 46.11s


--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 47: character maps to <undefined>
Call stack:
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\threading.py", line 973, in _bootstrap
    self._bootstrap_inner()
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\concurrent\future

## 8 · Step 3 — SRT Subtitle Generation

In [10]:
def generate_srt(script_data: dict, total_duration: float) -> Path:
    """
    Auto-generate SRT from script segments.
    Distributes time evenly proportional to word count.
    """
    segments = script_data['segments']
    word_counts = [len(s['text'].split()) for s in segments]
    total_words = sum(word_counts)

    srt_path = OUTPUT_DIR / f'subtitles_{TS}.srt'

    def fmt_time(s: float) -> str:
        h = int(s // 3600)
        m = int((s % 3600) // 60)
        sec = int(s % 60)
        ms = int((s - int(s)) * 1000)
        return f'{h:02d}:{m:02d}:{sec:02d},{ms:03d}'

    lines = []
    cursor = 0.0
    for i, (seg, wc) in enumerate(zip(segments, word_counts), start=1):
        duration = (wc / total_words) * total_duration
        end = cursor + duration
        lines.append(f"{i}\n{fmt_time(cursor)} --> {fmt_time(end)}\n{seg['text']}\n")
        cursor = end

    srt_path.write_text('\n'.join(lines))
    logger.info(f'SRT saved → {srt_path}')
    return srt_path

srt_path = generate_srt(script_data, script_data['duration_estimate'])
print(f'✅ SRT → {srt_path}')
print(srt_path.read_text())

2026-06-10 08:32:38,822 [INFO] SRT saved → output\subtitles_20260610_083137.srt
✅ SRT → output\subtitles_20260610_083137.srt
1
00:00:00,000 --> 00:00:10,952
Feeling overwhelmed by your codebase? What if I told you there are three simple habits that can totally transform your daily coding focus?

2
00:00:10,952 --> 00:00:25,238
First up, the Pomodoro Technique. Break your coding sessions into focused sprints. Work intensely for 25 minutes, then take a 5-minute break. This keeps your mind sharp and prevents burnout.

3
00:00:25,238 --> 00:00:46,190
Next, 'Eat the Frog'. Start your day by tackling your most challenging task first. Get that tough bug fix or complex feature out of the way when your energy and focus are highest. It sets a productive tone for the rest of your day.

4
00:00:46,190 --> 00:01:01,904
Finally, leverage keyboard shortcuts and automation tools. Seriously, minimize repetitive actions. Learn your IDE's shortcuts, set up custom snippets, or automate common Git commands

--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 41: character maps to <undefined>
Call stack:
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\deven\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\deven\AppData\Roaming\Python\Python310\site

## 9 · Step 4 — FFmpeg Assembly

In [11]:
!pip install imageio-ffmpeg
import imageio_ffmpeg

FFMPEG_BIN  = imageio_ffmpeg.get_ffmpeg_exe()
FFPROBE_BIN = str(Path(imageio_ffmpeg.get_ffmpeg_exe()).parent / 'ffprobe.exe')

In [12]:
import imageio_ffmpeg

FFMPEG_BIN  = imageio_ffmpeg.get_ffmpeg_exe()
FFPROBE_BIN = str(Path(FFMPEG_BIN).parent / 'ffprobe.exe')

@timed_step('Step 4 — FFmpeg Assembly')
def assemble_video(audio_path: Path,
                   clip_paths: list[Path],
                   srt_path: Path,
                   duration: float) -> Path:
    """
    1. Concatenate clips to fill audio duration
    2. Overlay voiceover as primary audio
    3. Burn-in subtitles
    Output: output/final_TIMESTAMP.mp4
    """
    final_path   = OUTPUT_DIR / f'final_{TS}.mp4'
    concat_list  = OUTPUT_DIR / f'concat_{TS}.txt'
    raw_video    = OUTPUT_DIR / f'raw_concat_{TS}.mp4'
    merged_video = OUTPUT_DIR / f'merged_{TS}.mp4'

    # ── Build concat list (loop clips to cover audio duration) ──
    clip_durations = []
    for cp in clip_paths:
        try:
            probe = subprocess.run(
                [FFPROBE_BIN, '-v', 'error', '-show_entries', 'format=duration',
                 '-of', 'csv=p=0', str(cp)],
                capture_output=True, text=True, check=True
            )
            clip_durations.append(float(probe.stdout.strip()))
        except Exception:
            clip_durations.append(10.0)  # assume 10s if probe fails

    concat_entries = []
    acc = 0.0
    idx = 0
    while acc < duration:
        cp = clip_paths[idx % len(clip_paths)]
        cd = clip_durations[idx % len(clip_durations)]
        concat_entries.append(f"file '{cp.resolve()}'")
        acc += cd
        idx += 1

    concat_list.write_text('\n'.join(concat_entries))

    def run(cmd, step_name):
        logger.info(f'FFmpeg [{step_name}]: {" ".join(cmd)}')
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            logger.error(f'FFmpeg stderr:\n{result.stderr}')
            log_error(f'ffmpeg_{step_name}',
                      RuntimeError(result.stderr[:500]))
            raise RuntimeError(f'FFmpeg failed at {step_name}')
        return result

    # ── 1. Concatenate clips ──────────────────────────────────
    run([
        FFMPEG_BIN, '-y', '-f', 'concat', '-safe', '0',
        '-i', str(concat_list),
        '-c', 'copy',
        str(raw_video)
    ], 'concat')

    # ── 2. Trim to audio duration + merge audio ───────────────
    run([
        FFMPEG_BIN, '-y',
        '-i', str(raw_video),
        '-i', str(audio_path),
        '-map', '0:v:0', '-map', '1:a:0',
        '-t', str(duration),
        '-c:v', 'libx264', '-crf', '23', '-preset', 'fast',
        '-c:a', 'aac', '-b:a', '192k',
        str(merged_video)
    ], 'merge_audio')

     # ── 3. Burn subtitles ─────────────────────────────────────
    srt_escaped = str(srt_path.resolve()).replace('\\', '/').replace(':', '\\:')
    run([
        FFMPEG_BIN, '-y',
        '-i', str(merged_video),
        '-vf', f"subtitles='{srt_escaped}':force_style='FontSize=10,PrimaryColour=&HFFFFFF,OutlineColour=&H000000,Outline=2,Alignment=2,MarginV=20'",
        '-c:a', 'copy',
        str(final_path)
    ], 'subtitles')
    
    # Cleanup intermediates
    for tmp in [concat_list, raw_video, merged_video]:
        tmp.unlink(missing_ok=True)

    logger.info(f'Final video → {final_path}')
    return final_path


# ── Run ──────────────────────────────────────────────────────
final_video_path = assemble_video(
    audio_path, clip_paths, srt_path,
    script_data['duration_estimate']
)
print(f'✅ Final video → {final_video_path}')

2026-06-10 08:32:40,320 [INFO] START  Step 4 — FFmpeg Assembly
2026-06-10 08:32:40,328 [INFO] FFmpeg [concat]: c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -f concat -safe 0 -i output\concat_20260610_083137.txt -c copy output\raw_concat_20260610_083137.mp4
2026-06-10 08:32:40,437 [INFO] FFmpeg [merge_audio]: c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -i output\raw_concat_20260610_083137.mp4 -i output\audio_20260610_083137.mp3 -map 0:v:0 -map 1:a:0 -t 70 -c:v libx264 -crf 23 -preset fast -c:a aac -b:a 192k output\merged_20260610_083137.mp4
2026-06-10 08:32:43,677 [INFO] FFmpeg [subtitles]: c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -i output\merged_20260610_083137.mp4 -vf subtitles='C\:/Users/deven/Desktop/Youtube_Automation/output/subtitles

--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 43: character maps to <undefined>
Call stack:
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\deven\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\deven\AppData\Roaming\Python\Python310\site

## 10 · Step 5 — YouTube Upload

In [13]:
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import pickle

YOUTUBE_SCOPES = ['https://www.googleapis.com/auth/youtube.upload']


def _get_youtube_service():
    """OAuth2 flow — opens browser on first run, caches token after."""
    token_path  = Path(YOUTUBE_TOKEN_PATH)
    creds       = None

    if token_path.exists():
        with open(token_path, 'rb') as f:
            creds = pickle.load(f)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                YOUTUBE_CLIENT_SECRETS_PATH, YOUTUBE_SCOPES)
            creds = flow.run_local_server(port=0)

        with open(token_path, 'wb') as f:
            pickle.dump(creds, f)

    return build('youtube', 'v3', credentials=creds)


def _build_description(script_data: dict, idea: dict) -> str:
    summary_points = ' | '.join(
        seg['text'][:80] + '…' for seg in script_data['segments']
        if seg['label'].startswith('point')
    )
    return (
        f"{idea['hook']}\n\n"
        f"In this video:\n{summary_points}\n\n"
        f"Keywords: {', '.join(idea['keywords'])}"
    )


@timed_step('Step 5 — YouTube Upload')
def upload_to_youtube(video_path: Path,
                      idea: dict,
                      script_data: dict,
                      max_retries: int = 2) -> str:
    youtube = _get_youtube_service()
    description = _build_description(script_data, idea)
    tags = list(dict.fromkeys(
        idea['keywords'] + [CONTENT_NICHE, TARGET_AUDIENCE.split()[0]]
    ))

    body = {
        'snippet': {
            'title':       idea['title'],
            'description': description,
            'tags':        tags,
            'categoryId':  '22',  # People & Blogs
        },
        'status': {
            'privacyStatus': YOUTUBE_PRIVACY_STATUS
        }
    }

    media = MediaFileUpload(str(video_path), mimetype='video/mp4', resumable=True)

    for attempt in range(1, max_retries + 2):
        try:
            logger.info(f'YouTube upload attempt {attempt}')
            request = youtube.videos().insert(
                part='snippet,status', body=body, media_body=media)
            response = None
            while response is None:
                _, response = request.next_chunk()

            video_id  = response['id']
            video_url = f'https://www.youtube.com/watch?v={video_id}'
            logger.info(f'Upload success → {video_url}')

            # Log to CSV
            log_csv = LOGS_DIR / 'upload_log.csv'
            write_header = not log_csv.exists()
            with open(log_csv, 'a', newline='') as f:
                w = csv.writer(f)
                if write_header:
                    w.writerow(['timestamp', 'video_id', 'url', 'title', 'privacy'])
                w.writerow([datetime.now().isoformat(), video_id,
                            video_url, idea['title'], YOUTUBE_PRIVACY_STATUS])

            return video_url

        except Exception as e:
            log_error('youtube_upload', e)
            if attempt <= max_retries:
                wait = 2 ** attempt
                logger.warning(f'Upload failed (attempt {attempt}). Retry in {wait}s…')
                time.sleep(wait)
            else:
                logger.error('YouTube upload failed after all retries.')
                raise

print('✅ Step 5 function defined')

✅ Step 5 function defined


c:\Users\deven\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [14]:
# ── Run upload ───────────────────────────────────────────────
# NOTE: First run will open a browser window for OAuth2 consent.
# Make sure credentials/client_secrets.json exists.

try:
    video_url = upload_to_youtube(final_video_path, idea, script_data)
    print(f'🎉 Uploaded! {video_url}')
except Exception as e:
    print(f'❌ Upload failed: {e}')
    print('   Check logs/error_log.txt for details.')

2026-06-10 08:32:47,671 [INFO] START  Step 5 — YouTube Upload
❌ Upload failed: [Errno 2] No such file or directory: 'credentials/client_secrets.json'
   Check logs/error_log.txt for details.


## 11 · Full Pipeline Runner (convenience cell)

In [15]:
def run_full_pipeline(
    topic: str  = TOPIC_SEED,
    niche: str  = CONTENT_NICHE,
    audience: str = TARGET_AUDIENCE,
    upload: bool  = True
) -> dict:
    """
    Run the entire pipeline end-to-end.
    Returns a summary dict with all output paths + video URL.
    """
    summary = {'status': 'started', 'ts': TS}

    try:
        # ── 1. Idea & Script (sequential) ────────────────────
        idea_data        = generate_idea(topic, niche, audience)
        script_result    = generate_script(idea_data)
        srt              = generate_srt(script_result, script_result['duration_estimate'])

        # ── 2. TTS + Clips (parallel) ─────────────────────────
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
            f_audio = ex.submit(generate_voiceover, script_result)
            f_clips = ex.submit(download_clips, idea_data['keywords'])
            aud   = f_audio.result()   # raises on failure → caught below
            clips = f_clips.result()

        if not clips:
            raise RuntimeError('No clips available — cannot assemble video')

        # ── 3. Assembly ───────────────────────────────────────
        final = assemble_video(aud, clips, srt, script_result['duration_estimate'])

        # ── 4. Upload ─────────────────────────────────────────
        url = None
        if upload:
            url = upload_to_youtube(final, idea_data, script_result)

        summary.update({
            'status':      'success',
            'idea_file':   str(OUTPUT_DIR / f'idea_{TS}.json'),
            'script_file': str(OUTPUT_DIR / f'script_{TS}.txt'),
            'audio_file':  str(aud),
            'clips':       [str(c) for c in clips],
            'video_file':  str(final),
            'youtube_url': url,
        })

    except Exception as e:
        log_error('full_pipeline', e)
        summary.update({'status': 'failed', 'error': str(e)})
        logger.error(f'Pipeline failed: {e}')

    print(json.dumps(summary, indent=2))
    return summary


# ── Uncomment to run the entire pipeline in one shot ─────────
# result = run_full_pipeline(upload=False)  # set upload=True when credentials are ready

## 12 · Diagnostics

In [16]:
# ── Check FFmpeg availability ─────────────────────────────────
import shutil

checks = {
    'ffmpeg':   shutil.which('ffmpeg'),
    'ffprobe':  shutil.which('ffprobe'),
}

for tool, path in checks.items():
    status = f'✅ {path}' if path else '❌ NOT FOUND — install ffmpeg'
    print(f'{tool:<12} {status}')

# ── Check API keys loaded ─────────────────────────────────────
key_checks = {
    'GEMINI_API_KEY':  GEMINI_API_KEY,
    'PEXELS_API_KEY':  PEXELS_API_KEY,
    'PIXABAY_API_KEY': PIXABAY_API_KEY,
}
print()
for name, val in key_checks.items():
    masked = val[:8] + '…' if len(val) > 8 else '(empty)'
    status = '✅' if val and val != f'YOUR_{name.split("_")[0]}_KEY_HERE' else '⚠️  NOT SET'
    print(f'{name:<25} {status}  {masked}')

# ── Check output dirs ─────────────────────────────────────────
print()
for d in [OUTPUT_DIR, CLIPS_DIR, FALLBACK_DIR, CREDENTIALS_DIR, LOGS_DIR]:
    exists = '✅' if d.exists() else '❌'
    print(f'{exists} {d}')

ffmpeg       ❌ NOT FOUND — install ffmpeg
ffprobe      ❌ NOT FOUND — install ffmpeg

GEMINI_API_KEY            ✅  AQ.Ab8RN…
PEXELS_API_KEY            ✅  qX6XGgEe…
PIXABAY_API_KEY           ⚠️  NOT SET  YOUR_PIX…

✅ output
✅ output\clips
✅ assets\fallback
✅ credentials
✅ logs


In [17]:
# ── Tail the pipeline log ─────────────────────────────────────
log_file = LOGS_DIR / 'pipeline_log.txt'
if log_file.exists():
    lines = log_file.read_text().splitlines()
    print(f'Last 30 lines of {log_file}:\n')
    print('\n'.join(lines[-30:]))
else:
    print('No pipeline log yet — run the pipeline first.')

Last 30 lines of logs\pipeline_log.txt:

2026-06-10 08:31:45,185 [INFO] Idea saved -> output\idea_20260610_083137.json
2026-06-10 08:31:45,190 [INFO] FINISH Step 1A - Idea Generation  (4.33s)
2026-06-10 08:31:45,240 [INFO] START  Step 1B — Script Generation
2026-06-10 08:31:45,249 [INFO] Gemini API call (attempt 1) temperature=0.5
2026-06-10 08:31:45,260 [INFO] AFC is enabled with max remote calls: 10.
2026-06-10 08:31:52,439 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 08:31:52,506 [INFO] FINISH Step 1B — Script Generation  (7.26s)
2026-06-10 08:31:52,684 [INFO] START Parallel -- TTS + Clip Downloads
2026-06-10 08:31:52,693 [INFO] START  Step 2A — Kokoro TTS
2026-06-10 08:31:52,702 [INFO] START  Step 2B -- Video Clips
2026-06-10 08:31:53,705 [INFO] Clip downloaded (pexels) -> output\clips\clip_dev_productivity_20260610_083137.mp4
2026-06-10 08:31:54,199 [INFO] Clip downloaded (pexels) ->